In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [ ]:
#load dataset and Fix issues
df=pd.read_csv("/Users/ajithsreepuram/Desktop/Customer-Churn predictor/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"]=pd.to_numeric(df["TotalCharges"],errors="coerce")
df["TotalCharges"]=df["TotalCharges"].fillna(df["TotalCharges"].median())
# df["Churn"]=df["Churn"].map({"Yes":1,"No":0})
print(f"Dataset shape: {df.shape}")
print(f"Missing values after fix:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
df.head()

Dataset shape: (7043, 21)
Missing values after fix:
Series([], dtype: int64)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
conn = sqlite3.connect("/Users/ajithsreepuram/Desktop/Customer-Churn predictor/data/churn.db")
df.to_sql("customers", conn, if_exists="replace", index=False)
print("Data loaded into SQLite successfully!")

Data loaded into SQLite successfully!


In [7]:
# SQL Query 1 — Overall churn rate
q1 = pd.read_sql("""
    SELECT 
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct
    FROM customers
""", conn)
print("Overall Churn Rate:")
print(q1)
# SQL Query 2 — Churn rate by Contract type
q2 = pd.read_sql("""
    SELECT 
        Contract,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_pct
    FROM customers
    GROUP BY Contract
    ORDER BY churn_pct DESC
""", conn)
print("Churn Rate by Contract Type:")
print(q2)
# SQL Query 3 — Avg MonthlyCharges: churned vs retained
q3 = pd.read_sql("""
    SELECT 
        Churn,
        ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges,
        ROUND(AVG(tenure), 2) AS avg_tenure_months
    FROM customers
    GROUP BY Churn
""", conn)
print("Avg Charges & Tenure by Churn:")
print(q3)

Overall Churn Rate:
   total_customers  churned  churn_rate_pct
0             7043     1869           26.54
Churn Rate by Contract Type:
         Contract  total_customers  churned  churn_pct
0  Month-to-month             3875     1655      42.71
1        One year             1473      166      11.27
2        Two year             1695       48       2.83
Avg Charges & Tenure by Churn:
  Churn  avg_monthly_charges  avg_tenure_months
0    No                61.27              37.57
1   Yes                74.44              17.98


In [8]:
# SQL Query 4 — Churn by tenure bucket
q4 = pd.read_sql("""
    SELECT 
        CASE 
            WHEN tenure <= 12  THEN '0-12 months'
            WHEN tenure <= 24  THEN '13-24 months'
            WHEN tenure <= 48  THEN '25-48 months'
            ELSE '48+ months'
        END AS tenure_bucket,
        COUNT(*) AS total,
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_pct
    FROM customers
    GROUP BY tenure_bucket
    ORDER BY churn_pct DESC
""", conn)
print("Churn by Tenure Bucket:")
print(q4)
# SQL Query 5 — Churn by InternetService
q5 = pd.read_sql("""
    SELECT InternetService,
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_pct
    FROM customers GROUP BY InternetService ORDER BY churn_pct DESC
""", conn)
print("Query 5 — Churn by Internet Service:"); print(q5)

# SQL Query 6 — Churn by PaymentMethod
q6 = pd.read_sql("""
    SELECT PaymentMethod,
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_pct
    FROM customers GROUP BY PaymentMethod ORDER BY churn_pct DESC
""", conn)
print("\nQuery 6 — Churn by Payment Method:"); print(q6)

# SQL Query 7 — Senior Citizens churn rate
q7 = pd.read_sql("""
    SELECT SeniorCitizen,
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_pct
    FROM customers GROUP BY SeniorCitizen
""", conn)
print("\nQuery 7 — Senior Citizen Churn:"); print(q7)

# SQL Query 8 — Top 5 highest-value churned customers
q8 = pd.read_sql("""
    SELECT customerID, MonthlyCharges, tenure, Contract
    FROM customers WHERE Churn = 'Yes'
    ORDER BY MonthlyCharges DESC LIMIT 5
""", conn)
print("\nQuery 8 — Top 5 Highest-Value Churned Customers:"); print(q8)

# SQL Query 9 — No support services churn
q9 = pd.read_sql("""
    SELECT 
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_pct
    FROM customers
    WHERE TechSupport='No' AND OnlineSecurity='No'
""", conn)
print("\nQuery 9 — Churn rate with no support services:"); print(q9)

# SQL Query 10 — Churn by number of streaming services
q10 = pd.read_sql("""
    SELECT 
        (CASE WHEN StreamingTV='Yes' THEN 1 ELSE 0 END +
         CASE WHEN StreamingMovies='Yes' THEN 1 ELSE 0 END) AS streaming_services,
        ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_pct
    FROM customers
    GROUP BY streaming_services ORDER BY streaming_services
""", conn)
print("\nQuery 10 — Churn by Streaming Services count:"); print(q10)


Churn by Tenure Bucket:
  tenure_bucket  total  churn_pct
0   0-12 months   2186      47.44
1  13-24 months   1024      28.71
2  25-48 months   1594      20.39
3    48+ months   2239       9.51
Query 5 — Churn by Internet Service:
  InternetService  churn_pct
0     Fiber optic      41.89
1             DSL      18.96
2              No       7.40

Query 6 — Churn by Payment Method:
               PaymentMethod  churn_pct
0           Electronic check      45.29
1               Mailed check      19.11
2  Bank transfer (automatic)      16.71
3    Credit card (automatic)      15.24

Query 7 — Senior Citizen Churn:
   SeniorCitizen  churn_pct
0              0      23.61
1              1      41.68

Query 8 — Top 5 Highest-Value Churned Customers:
   customerID  MonthlyCharges  tenure        Contract
0  8199-ZLLSA          118.35      67        One year
1  2889-FPWRM          117.80      72        One year
2  2302-ANTDP          117.45      48  Month-to-month
3  9053-JZFKV          116.20     

In [9]:
# Feature Engineering 
df['avg_monthly_spend']   = df['TotalCharges'] / (df['tenure'] + 1)
df['services_count']      = df[['PhoneService', 'InternetService',
                                  'OnlineSecurity', 'TechSupport']].apply(
                             lambda x: (x != 'No').sum(), axis=1)
df['is_new_customer']     = (df['tenure'] <= 12).astype(int)
df['high_value_customer'] = (df['MonthlyCharges'] > df['MonthlyCharges'].quantile(0.75)).astype(int)

print("New features added:")
print(df[['tenure', 'TotalCharges', 'avg_monthly_spend',
          'services_count', 'is_new_customer', 'high_value_customer']].head(10))


New features added:
   tenure  TotalCharges  avg_monthly_spend  services_count  is_new_customer  \
0       1         29.85          14.925000               1                1   
1      34       1889.50          53.985714               3                0   
2       2        108.15          36.050000               3                1   
3      45       1840.75          40.016304               3                0   
4       2        151.65          50.550000               2                1   
5       8        820.50          91.166667               2                1   
6      22       1949.40          84.756522               2                0   
7      10        301.90          27.445455               2                1   
8      28       3046.05         105.036207               3                0   
9      62       3487.95          55.364286               3                0   

   high_value_customer  
0                    0  
1                    0  
2                    0  
3         

In [10]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Separate features and target
X = df.drop(["Churn", "customerID"], axis=1)
Y = df["Churn"]

print(f"Features shape: {X.shape}")
print(f"Churn distribution:\n{Y.value_counts(normalize=True).round(3)}")

Features shape: (7043, 23)
Churn distribution:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


In [11]:
# Binary columns — Label encode (Yes/No → 1/0)
X["gender"]           = X["gender"].map({"Male": 1, "Female": 0})
X["Partner"]          = X["Partner"].map({"Yes": 1, "No": 0})
X["Dependents"]       = X["Dependents"].map({"Yes": 1, "No": 0})
X["PhoneService"]     = X["PhoneService"].map({"Yes": 1, "No": 0})
X["PaperlessBilling"] = X["PaperlessBilling"].map({"Yes": 1, "No": 0})
# SeniorCitizen is already 0/1, no change needed

# Multi-class columns — One-Hot encode
multi_cols = ["MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
              "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
              "Contract", "PaymentMethod"]
X = pd.get_dummies(X, columns=multi_cols, drop_first=True)

print(f"Shape after encoding: {X.shape}")
print(f"Columns: {list(X.columns)}")

Shape after encoding: (7043, 34)
Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'services_count', 'is_new_customer', 'high_value_customer', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [12]:
from sklearn.model_selection import train_test_split

# stratify=Y ensures both train and test have same churn ratio
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Train churn rate: {Y_train.mean():.3f} | Test churn rate: {Y_test.mean():.3f}")


Train size: 5634 | Test size: 1409
Train churn rate: 0.265 | Test churn rate: 0.265


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# fit_transform on train — learn mean/std from train only
X_train_scaled = scaler.fit_transform(X_train)
# transform on test — use same mean/std from train
X_test_scaled  = scaler.transform(X_test)

print("Scaling done. Data is ready for modeling.")

Scaling done. Data is ready for modeling.


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, Y_train)

Y_pred = lr_model.predict(X_test_scaled)
Y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression (Baseline) ===")
print(classification_report(Y_test, Y_pred))
print(f"ROC-AUC Score: {roc_auc_score(Y_test, Y_prob):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, Y_pred))

=== Logistic Regression (Baseline) ===
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.67      0.53      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.80      0.80      0.80      1409

ROC-AUC Score: 0.8460

Confusion Matrix:
[[936  99]
 [176 198]]


In [15]:
import pickle, os

save_path = "/Users/ajithsreepuram/Desktop/Customer-Churn predictor/data/"

# Save train/test splits (unscaled — for SHAP later)
X_train.to_csv(save_path + "X_train.csv", index=False)
X_test.to_csv(save_path  + "X_test.csv",  index=False)
Y_train.to_csv(save_path + "Y_train.csv", index=False)
Y_test.to_csv(save_path  + "Y_test.csv",  index=False)

# Save scaler
pickle.dump(scaler, open(save_path + "scaler.pkl", "wb"))

print("Saved: X_train, X_test, Y_train, Y_test, scaler.pkl")
print("Next step: open 02_modeling.ipynb")

Saved: X_train, X_test, Y_train, Y_test, scaler.pkl
Next step: open 02_modeling.ipynb
